In [1]:
pip install torch torchvision matplotlib

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import sys
import subprocess

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

trainset = torchvision.datasets.CIFAR10(root='./data', train=True,
                                        download=True, transform=transform)

testset = torchvision.datasets.CIFAR10(root='./data', train=False,
                                       download=True, transform=transform)

trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

100%|██████████| 170M/170M [00:02<00:00, 70.6MB/s] 


In [5]:
class ChannelAttention(nn.Module):
    def __init__(self, in_channels, ratio=16):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_channels, in_channels // ratio),
            nn.ReLU(),
            nn.Linear(in_channels // ratio, in_channels)
        )
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, h, w = x.size()

        avg_pool = torch.mean(x, dim=(2,3))
        max_pool = torch.amax(x, dim=(2,3))

        avg_out = self.mlp(avg_pool)
        max_out = self.mlp(max_pool)

        out = avg_out + max_out
        return self.sigmoid(out).view(b, c, 1, 1)

In [6]:
class SpatialAttention(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Conv2d(2, 1, kernel_size=7, padding=3)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        avg_pool = torch.mean(x, dim=1, keepdim=True)
        max_pool = torch.amax(x, dim=1, keepdim=True)

        x = torch.cat([avg_pool, max_pool], dim=1)
        x = self.conv(x)
        return self.sigmoid(x)

In [7]:
class CBAM(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.ca = ChannelAttention(channels)
        self.sa = SpatialAttention()

    def forward(self, x):
        x = self.ca(x) * x
        x = self.sa(x) * x
        return x

In [8]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        out += self.shortcut(x)
        out = self.relu(out)

        return out

In [9]:
class BasicBlockCBAM(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        self.cbam = CBAM(out_channels)   # 🔥 added

        self.relu = nn.ReLU(inplace=True)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        out = self.cbam(out)   # 🔥 apply CBAM

        out += self.shortcut(x)
        out = self.relu(out)

        return out

In [10]:
class ResNet(nn.Module):
    def __init__(self, block, num_classes=10):
        super().__init__()

        self.in_channels = 64

        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(block, 64, 2, stride=1)
        self.layer2 = self._make_layer(block, 128, 2, stride=2)
        self.layer3 = self._make_layer(block, 256, 2, stride=2)
        self.layer4 = self._make_layer(block, 512, 2, stride=2)

        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, block, out_channels, blocks, stride):
        layers = []

        layers.append(block(self.in_channels, out_channels, stride))
        self.in_channels = out_channels

        for _ in range(1, blocks):
            layers.append(block(out_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)

        return x

In [11]:
def train(model, loader, optimizer, criterion, device):
    model.train()
    total_loss = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(loader)

In [12]:
# baseline_model = SimpleCNN(use_cbam=False).to(device)
# cbam_model = SimpleCNN(use_cbam=True).to(device)
# torch.save(baseline_model.state_dict(), "baseline.pth")
# torch.save(cbam_model.state_dict(), "cbam.pth")

In [13]:
def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    return 100 * correct / total

In [14]:
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Baseline ResNet18
baseline_model = ResNet(BasicBlock).to(device)

# CBAM ResNet18
cbam_model = ResNet(BasicBlockCBAM).to(device)

torch.save(baseline_model.state_dict(), "baseline.pth")
torch.save(cbam_model.state_dict(), "cbam.pth")

criterion = nn.CrossEntropyLoss()

optimizer1 = optim.Adam(baseline_model.parameters(), lr=0.0005)
optimizer2 = optim.Adam(cbam_model.parameters(), lr=0.0005)

epochs = 30

print("Training Baseline...")
for epoch in range(epochs):
    loss = train(baseline_model, trainloader, optimizer1, criterion, device)
    acc = evaluate(baseline_model, testloader, device)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}, Accuracy: {acc:.2f}%")

print("\nTraining CBAM...")
for epoch in range(epochs):
    loss = train(cbam_model, trainloader, optimizer2, criterion, device)
    acc = evaluate(cbam_model, testloader, device)
    print(f"Epoch {epoch+1}, Loss: {loss:.4f}, Accuracy: {acc:.2f}%")

Training Baseline...
Epoch 1, Loss: 1.2175, Accuracy: 61.24%
Epoch 2, Loss: 0.7332, Accuracy: 76.64%
Epoch 3, Loss: 0.5435, Accuracy: 77.69%
Epoch 4, Loss: 0.4260, Accuracy: 80.34%
Epoch 5, Loss: 0.3314, Accuracy: 81.85%
Epoch 6, Loss: 0.2447, Accuracy: 82.22%
Epoch 7, Loss: 0.1792, Accuracy: 82.19%
Epoch 8, Loss: 0.1327, Accuracy: 83.22%
Epoch 9, Loss: 0.1018, Accuracy: 83.24%
Epoch 10, Loss: 0.0830, Accuracy: 82.99%
Epoch 11, Loss: 0.0702, Accuracy: 83.79%
Epoch 12, Loss: 0.0630, Accuracy: 84.20%
Epoch 13, Loss: 0.0546, Accuracy: 83.07%
Epoch 14, Loss: 0.0493, Accuracy: 83.86%
Epoch 15, Loss: 0.0479, Accuracy: 84.41%
Epoch 16, Loss: 0.0374, Accuracy: 83.47%
Epoch 17, Loss: 0.0419, Accuracy: 84.84%
Epoch 18, Loss: 0.0338, Accuracy: 84.54%
Epoch 19, Loss: 0.0367, Accuracy: 84.25%
Epoch 20, Loss: 0.0333, Accuracy: 83.21%
Epoch 21, Loss: 0.0324, Accuracy: 84.06%
Epoch 22, Loss: 0.0258, Accuracy: 84.60%
Epoch 23, Loss: 0.0263, Accuracy: 82.64%
Epoch 24, Loss: 0.0265, Accuracy: 84.03%
Epoc

In [ ]:


subprocess.run([
    'git', 'clone', '--quiet',
    'https://github.com/Jongchan/attention-module.git',
    '/tmp/official_cbam'
], check=True)

CompletedProcess(args=['git', 'clone', '--quiet', 'https://github.com/Jongchan/attention-module.git', '/tmp/official_cbam'], returncode=0)

In [6]:

sys.path.insert(0, '/tmp/official_cbam/MODELS')

from cbam import CBAM as OfficialCBAM
    

In [7]:
import torch.nn as nn

class BasicBlockCBAM_Official(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        self.conv1 = nn.Conv2d(in_channels, out_channels, 3, stride, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(out_channels)

        self.conv2 = nn.Conv2d(out_channels, out_channels, 3, 1, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(out_channels)

        # 🔥 official CBAM
        self.cbam = OfficialCBAM(out_channels, reduction_ratio=16)

        self.relu = nn.ReLU(inplace=True)

        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        identity = x

        out = self.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))

        out = self.cbam(out)   # 🔥 official CBAM applied

        out += self.shortcut(x)
        out = self.relu(out)

        return out

In [8]:
class ResNetOfficial(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()

        self.in_channels = 64

        self.conv1 = nn.Conv2d(3, 64, 3, 1, 1, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)

        self.layer1 = self._make_layer(64, 2, stride=1)
        self.layer2 = self._make_layer(128, 2, stride=2)
        self.layer3 = self._make_layer(256, 2, stride=2)
        self.layer4 = self._make_layer(512, 2, stride=2)

        self.pool = nn.AdaptiveAvgPool2d((1,1))
        self.fc = nn.Linear(512, num_classes)

    def _make_layer(self, out_channels, blocks, stride):
        layers = []
        layers.append(BasicBlockCBAM_Official(self.in_channels, out_channels, stride))
        self.in_channels = out_channels

        for _ in range(1, blocks):
            layers.append(BasicBlockCBAM_Official(out_channels, out_channels))

        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))

        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)

        x = self.pool(x)
        x = x.view(x.size(0), -1)

        return self.fc(x)

In [9]:
official_cbam_model = ResNetOfficial().to(device)
optimizer_official = torch.optim.Adam(
    official_cbam_model.parameters(), lr=0.0005
)

In [10]:
# Loss function
criterion = nn.CrossEntropyLoss()

# Optimizer (same as your setup)
optimizer_official = torch.optim.Adam(
    official_cbam_model.parameters(), lr=0.0005
)

epochs = 30

print("Training Official CBAM...")

for epoch in range(epochs):
    official_cbam_model.train()
    total_loss = 0

    for images, labels in trainloader:
        images, labels = images.to(device), labels.to(device)

        optimizer_official.zero_grad()
        outputs = official_cbam_model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer_official.step()

        total_loss += loss.item()

    # Evaluation
    official_cbam_model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in testloader:
            images, labels = images.to(device), labels.to(device)
            outputs = official_cbam_model(images)
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    acc = 100 * correct / total
    avg_loss = total_loss / len(trainloader)

    print(f"Epoch {epoch+1}, Loss: {avg_loss:.4f}, Accuracy: {acc:.2f}%")

Training Official CBAM...
Epoch 1, Loss: 1.1634, Accuracy: 64.77%
Epoch 2, Loss: 0.7347, Accuracy: 73.01%
Epoch 3, Loss: 0.5311, Accuracy: 74.09%
Epoch 4, Loss: 0.3924, Accuracy: 79.31%
Epoch 5, Loss: 0.2765, Accuracy: 78.47%
Epoch 6, Loss: 0.1892, Accuracy: 80.97%
Epoch 7, Loss: 0.1363, Accuracy: 79.85%
Epoch 8, Loss: 0.0940, Accuracy: 81.11%
Epoch 9, Loss: 0.0848, Accuracy: 80.89%
Epoch 10, Loss: 0.0649, Accuracy: 80.22%
Epoch 11, Loss: 0.0648, Accuracy: 80.45%
Epoch 12, Loss: 0.0528, Accuracy: 80.63%
Epoch 13, Loss: 0.0534, Accuracy: 80.51%
Epoch 14, Loss: 0.0469, Accuracy: 80.79%
Epoch 15, Loss: 0.0476, Accuracy: 81.40%
Epoch 16, Loss: 0.0391, Accuracy: 82.38%
Epoch 17, Loss: 0.0358, Accuracy: 81.20%
Epoch 18, Loss: 0.0382, Accuracy: 81.14%
Epoch 19, Loss: 0.0294, Accuracy: 81.67%
Epoch 20, Loss: 0.0328, Accuracy: 81.95%
Epoch 21, Loss: 0.0293, Accuracy: 81.61%
Epoch 22, Loss: 0.0304, Accuracy: 82.12%
Epoch 23, Loss: 0.0269, Accuracy: 81.89%
Epoch 24, Loss: 0.0267, Accuracy: 81.45%